# Notebook 01 — Ingesta Multiformato (Capa Bronze) y Análisis Exploratorio (EDA) en Databricks
**Proyecto:** Implementación de Soluciones BI, Big Data y Experimentación — Unity Perú SAC
**Curso:** Base de Datos Avanzadas y Big Data (NRC 26055) · Periodo 2026-2
**Arquitectura:** Medallion Lakehouse (Capa Bronze) sobre Apache Spark / Delta Lake

### Propósito del Notebook
1. Ingestar las **cuatro fuentes heterogéneas** del almacén de componentes tecnológicos en **DataFrames de Apache Spark**:
   - Movimientos y catálogo operacional provenientes de `ALMACEN_ANONIMO` (5,348 hechos, 24 componentes, 3,685 lotes).
   - Planificación mensual de demanda (`Plan_Demanda_2026`, 63 registros).
   - Serie horaria de sensores IoT ambientales (`Sensores_Almacen_2026.txt`, 25,824 lecturas).
   - Auditoría de conteos cíclicos (`Conteos_Ciclicos_2026.txt`, 72 registros).
2. Persistir los datasets crudos con metadatos de linaje (`_fuente`, `_ts_ingesta`) en la **Capa Bronze** (Delta / Parquet).
3. Ejecutar el **Análisis Exploratorio de Datos (EDA)** distribuido (`printSchema`, `describe`, conteo de valores nulos y distribución por tipo de movimiento).


In [ ]:
# Configuración compatible tanto con Databricks Runtime como con ejecución PySpark local
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("UnityPeru_01_Bronze_EDA").getOrCreate()

# Ruta base: en Databricks ajustar a '/Volumes/workspace/default/unity_datos' o 'dbfs:/FileStore/unity_peru'
BASE_DATOS = "dbfs:/FileStore/unity_peru/datos"
BASE_RESPALDO = "dbfs:/FileStore/unity_peru/respaldo"
FORMATO_TABLA = "delta"  # Cambiar a 'parquet' si se ejecuta fuera de Databricks

print(f"Sesión iniciada correctamente — Apache Spark {spark.version}")


## 1. Carga de Fuentes en DataFrames de Apache Spark y Escritura en Capa Bronze


In [ ]:
# 1.1 Lectura de fuentes operacionales y complementarias en DataFrames de Spark
df_mov_raw = (
    spark.read.option("header", "true").option("sep", ";").option("inferSchema", "true")
    .csv(f"{BASE_RESPALDO}/Fact_Movimiento.csv")
    .withColumn("_fuente", F.lit("OLTP_ALMACEN_ANONIMO"))
    .withColumn("_ts_ingesta", F.current_timestamp())
)

df_prod_raw = (
    spark.read.option("header", "true").option("sep", ";").option("inferSchema", "true")
    .csv(f"{BASE_RESPALDO}/Dim_Producto.csv")
    .withColumn("_fuente", F.lit("OLTP_ALMACEN_ANONIMO"))
    .withColumn("_ts_ingesta", F.current_timestamp())
)

df_tipo_raw = (
    spark.read.option("header", "true").option("sep", ";").option("inferSchema", "true")
    .csv(f"{BASE_RESPALDO}/Dim_TipoMovimiento.csv")
)

df_proy_raw = (
    spark.read.option("header", "true").option("sep", ";").option("inferSchema", "true")
    .csv(f"{BASE_RESPALDO}/Dim_Proyecto.csv")
)

df_plan_raw = (
    spark.read.option("header", "true").option("sep", ";").option("inferSchema", "true")
    .csv(f"{BASE_RESPALDO}/stg_Plan_Mensual.csv")
    .withColumn("_fuente", F.lit("Plan_Demanda_2026.xlsx"))
    .withColumn("_ts_ingesta", F.current_timestamp())
)

df_sensores_raw = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(f"{BASE_DATOS}/Sensores_Almacen_2026.txt")
    .withColumn("_fuente", F.lit("Sensores_Almacen_2026.txt"))
    .withColumn("_ts_ingesta", F.current_timestamp())
)

df_conteos_raw = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(f"{BASE_DATOS}/Conteos_Ciclicos_2026.txt")
    .withColumn("_fuente", F.lit("Conteos_Ciclicos_2026.txt"))
    .withColumn("_ts_ingesta", F.current_timestamp())
)

# 1.2 Persistencia como tablas administradas de la Capa Bronze en Databricks
spark.sql("CREATE SCHEMA IF NOT EXISTS unity_bronze")
df_mov_raw.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_bronze.movimientos_raw")
df_prod_raw.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_bronze.productos_raw")
df_tipo_raw.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_bronze.tipos_movimiento_raw")
df_proy_raw.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_bronze.proyectos_raw")
df_plan_raw.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_bronze.plan_demanda_raw")
df_sensores_raw.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_bronze.sensores_iot_raw")
df_conteos_raw.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_bronze.conteos_ciclicos_raw")

print("Conteos Capa Bronze:")
for nombre, df_b in [
    ("unity_bronze.movimientos_raw", df_mov_raw),
    ("unity_bronze.productos_raw", df_prod_raw),
    ("unity_bronze.plan_demanda_raw", df_plan_raw),
    ("unity_bronze.sensores_iot_raw", df_sensores_raw),
    ("unity_bronze.conteos_ciclicos_raw", df_conteos_raw),
]:
    print(f"  {nombre:<36}: {df_b.count():>7,} registros")


## 2. Análisis Exploratorio de Datos (EDA) con PySpark (`describe`, perfil de nulos y distribuciones)


In [ ]:
# 2.1 Estadísticos descriptivos de las variables numéricas de movimientos
print("=== Estadísticos descriptivos: Movimientos de Almacén ===")
df_mov_raw.select("Cantidad", "Costo_Total", "Dias_Permanencia").describe().show()

print("=== Estadísticos descriptivos: Sensores Ambientales IoT ===")
df_sensores_raw.select("temperatura_c", "humedad_pct").describe().show()


In [ ]:
# 2.2 Auditoría de valores nulos por columna (interpretación semántica de nulidad)
cols_audit = ["Id_Movimiento_NK", "Id_Fecha", "Id_Producto", "Id_TipoMovimiento",
              "Id_Proyecto", "Id_Proveedor", "Cantidad", "Costo_Total", "Dias_Permanencia"]

df_nulos = df_mov_raw.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in cols_audit
])
df_nulos.show()

# 2.3 Distribución de movimientos y unidades por tipo operacional
(
    df_mov_raw.join(df_tipo_raw, on="Id_TipoMovimiento", how="left")
    .groupBy("Codigo", "Es_Entrada", "Es_Salida_Real")
    .agg(
        F.count("*").alias("movimientos"),
        F.sum("Cantidad").alias("unidades_totales"),
        F.round(F.sum("Costo_Total"), 2).alias("costo_total_soles"),
    )
    .show()
)
